In [ ]:
# BILSTM部分:
# 数据生成类:
import torch
from torch.utils.data import Dataset
from torch.nn import TransformerEncoder, TransformerEncoderLayer, LayerNorm
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaModel
import pandas as pd

class RobertaTextDataset(Dataset):
    def __init__(self, csv_file, max_length=128, device='cpu'):
        self.device = device
        self.data = pd.read_csv(csv_file)
        self.tokenizer = RobertaTokenizer.from_pretrained('/root/autodl-tmp/huggingface/roberta-base')
        self.model = RobertaModel.from_pretrained('/root/autodl-tmp/huggingface/roberta-base').to(device)
        self.model.eval()

        self.max_length = max_length
        self.proj_concat = nn.Linear(3*768, 768).to(device)

        encoder_layer = TransformerEncoderLayer(d_model=768, nhead=8).to(device)
        self.transformer = TransformerEncoder(encoder_layer, num_layers=1).to(device)
        self.norm_single = LayerNorm(768).to(device)   # 用于每个 embedding（title、summary、synopsis）
        self.norm_concat = LayerNorm(3*768).to(device)   # 用于 concat 后的多模态向量

        self.titleembedding = []
        self.summariedembedding = []
        self.synopsisembedding = []
        self.titleembeddingaftertf = []
        self.summariedembeddingaftertf = []
        self.synopsisembeddingaftertf = []
        self.multiembedding = []

        # 27类标签字段名
        self.label_columns = [
            "Drama", "Thriller", "Comedy", "Action", "Adventure", "Crime", "Romance", "Mystery", "Sci-Fi", "Fantasy",
            "Horror", "Dark Comedy", "Family", "Period Drama", "Biography", "Animation", "Romantic Comedy", "Tragedy",
            "Psychological Thriller", "Psychological Drama", "Supernatural Horror", "Slapstick", "War", "History",
            "Coming-of-Age", "Superhero", "Docudrama"
        ]

        self._preprocess_all()

    def _get_embedding(self, text):
        with torch.no_grad():
            inputs = self.tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=self.max_length).to(self.device)
            outputs = self.model(**inputs)
            return outputs.last_hidden_state.mean(dim=1).squeeze(0)

    def _transform_and_norm(self, embedding_list):
        batch_tensor = torch.stack(embedding_list).unsqueeze(1)
        transformed = self.transformer(batch_tensor).squeeze(1)
        return [self.norm_single(tensor) for tensor in transformed]

    def _preprocess_all(self):
        print("开始编码文本...")

        for _, row in self.data.iterrows():
            self.titleembedding.append(self._get_embedding(str(row['title'])))
            self.summariedembedding.append(self._get_embedding(str(row['summaries'])))
            self.synopsisembedding.append(self._get_embedding(str(row['synopsis'])))

        print("Transformer处理...")
        self.titleembeddingaftertf = self._transform_and_norm(self.titleembedding)
        self.summariedembeddingaftertf = self._transform_and_norm(self.summariedembedding)
        self.synopsisembeddingaftertf = self._transform_and_norm(self.synopsisembedding)

        print("生成最终多模态embedding...")
        for t, s, y in zip(self.titleembeddingaftertf, self.summariedembeddingaftertf, self.synopsisembeddingaftertf):
            concat = torch.cat([t, s, y], dim=0)
            reduced = self.proj_concat(concat)
            self.multiembedding.append(self.norm_single(reduced))  # norm over 1024
        self.labels = torch.tensor(self.data[self.label_columns].values, dtype=torch.float32).to(self.device)
        print("预处理完成.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]

        # 从缓存或重新计算得到 embedding
        title_embedding = torch.tensor(self.titleembedding[idx])           # (1024,)
        summaried_embedding = torch.tensor(self.summariedembedding[idx])   # (1024,)
        synopsis_embedding = torch.tensor(self.synopsisembedding[idx])     # (1024,)
        title_tf = torch.tensor(self.titleembeddingaftertf[idx])                          # (1024,)
        summaried_tf = torch.tensor(self.summariedembeddingaftertf[idx])                  # (1024,)
        synopsis_tf = torch.tensor(self.synopsisembeddingaftertf[idx])                    # (1024,)
        multi_embedding = torch.tensor(self.multiembedding[idx])            # (1024,)

        # 选 7 个组成最终 tensor
        emb_list = [
            title_embedding,
            summaried_embedding,
            synopsis_embedding,
            title_tf,
            summaried_tf,
            synopsis_tf,
            multi_embedding
        ]
        embeddings = torch.stack(emb_list)   # (6, 1024)

        label = torch.tensor(self.labels[idx], dtype=torch.float32)  # (27,)

        return {
            'embeddings': embeddings,
            'labels': label
        }

# 模型：
import torch
import torch.nn as nn

class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=512, num_layers=1, num_classes=27, dropout=0.3):
        super(BiLSTMClassifier, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0  # dropout 仅当 num_layers > 1 时有效
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)  # 输出 27 维（多标签 logits）
        )

    def forward(self, x):
        # x: (batch, 7, 768)
        lstm_out, _ = self.lstm(x)  # -> (batch, 7, hidden_dim * 2)
        pooled = torch.mean(lstm_out, dim=1)  # -> (batch, hidden_dim * 2)
        logits = self.classifier(pooled)      # -> (batch, 27)
        return logits  # 不加激活函数，留给 BCEWithLogitsLoss

# 训练脚本：
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from Model.BILSTMModel import BiLSTMClassifier
from TorchDataset.RobertaTextDataset import RobertaTextDataset

def collate_fn(batch):
    all_inputs = []
    all_labels = []

    for item in batch:
        # 拼接6个embedding向量 → 每个样本为 (7, 768)
        emb_list = [
            item['title_embedding'],
            item['summaried_embedding'],
            item['synopsis_embedding'],
            item['title_tf'],
            item['summaried_tf'],
            item['synopsis_tf'],
            item['multi_embedding']
        ]
        emb_tensor = torch.stack(emb_list)
        all_inputs.append(emb_tensor)

        # 使用真实的 27 维标签向量
        all_labels.append(item['label'])

    inputs_tensor = torch.stack(all_inputs)
    labels_tensor = torch.stack(all_labels)

    return inputs_tensor, labels_tensor


def save_model(model, path='bilstm_model.pt'):
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_class': model.__class__.__name__
    }, path)
    print(f"模型已保存到: {path}")


def train_model(dataset, device='cuda', num_epochs=5, batch_size=8, lr=1e-4, save_path='../weights/bilstm_model.pt'):
    model = BiLSTMClassifier(num_classes=27).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    best_loss = float('inf')

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0

        for batch_inputs, batch_labels in dataloader:
            batch_inputs, batch_labels = batch_inputs.to(device), batch_labels.to(device)

            outputs = model(batch_inputs)  # (batch, 27)
            loss = criterion(outputs, batch_labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{num_epochs} - Avg Loss: {avg_loss:.4f}")

        # 保存最优模型（按 loss）
        if avg_loss < best_loss:
            best_loss = avg_loss
            save_model(model, save_path)

if __name__ == '__main__':
    path = '/root/autodl-tmp/MovieLabeling/datasets/train_data.csv'
    dataset = RobertaTextDataset(csv_file=path, device='cuda')
train_model(dataset)

# 测试脚本:
import torch
import torch.nn as nn

import sys
import os
from torch.utils.data import DataLoader
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from Model.BILSTMModel import BiLSTMClassifier
from TorchDataset.RobertaTextDataset import RobertaTextDataset
from metrixCaler import evaluate_metrics


def collate_fn(batch):
    all_inputs = []
    all_labels = []

    for item in batch:
        emb_list = [
            item['title_embedding'],
            item['summaried_embedding'],
            item['synopsis_embedding'],
            item['title_tf'],
            item['summaried_tf'],
            item['synopsis_tf'],
            item['multi_embedding'],
        ]
        emb_tensor = torch.stack(emb_list)
        all_inputs.append(emb_tensor)

        all_labels.append(item['label'])  # 多标签向量

    inputs_tensor = torch.stack(all_inputs)  # (batch, 6, 1024)
    labels_tensor = torch.stack(all_labels)  # (batch, 27)
    return inputs_tensor, labels_tensor

def load_model(path='bilstm_model.pt', device='cpu'):
    checkpoint = torch.load(path, map_location=device)

    if checkpoint['model_class'] == 'BiLSTMClassifier':
        model = BiLSTMClassifier(num_classes=27).to(device)
    else:
        raise ValueError(f"未知模型类型: {checkpoint['model_class']}")

    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"模型已从 {path} 加载")
    return model

def test_model(model, dataloader, device='cuda'):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)  # logits
            probs = torch.sigmoid(outputs)

            preds = (probs > 0.5).float()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    # === 计算 Exact Match Accuracy ===

    correct_per_sample = (all_preds == all_labels).sum(dim=1)  # 每个样本27个label中对了多少个
    accuracy_per_sample = correct_per_sample / all_labels.size(1)  # 除以27
    mean_accuracy = accuracy_per_sample.mean().item()  # 所有样本平均准确率
    print(f"\nExample-based 多标签准确率 (mean sample-wise ACC): {mean_accuracy:.4f}")

    acc = 0#(preds == labels).float().mean(dim=0).mean().item()
    metrix = evaluate_metrics(all_preds,all_labels)
    print(metrix)
    acc = metrix['accuracy']

    print("\n预测完成！展示前5条预测结果：")
    for i in range(min(5, all_preds.size(0))):
        print(f"预测: {all_preds[i].tolist()}")
        print(f"真实: {all_labels[i].tolist()}")
        print("---")

    return all_preds, all_labels, exact_acc


if __name__ == '__main__':
     # === 计算准确率 ===

    path = '/root/autodl-tmp/MovieLabeling/datasets/test_data.csv'

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 构造数据集和模型
    dataset = RobertaTextDataset(csv_file=path, device=device)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

    model = load_model('../weights/bilstm_model_0.9.pt', device=device)

    # 运行测试
    all_preds, all_labels, acc = test_model(model, dataloader, device=device)

# RoBERTa部分:
# 数据生成:
import torch
from torch.utils.data import Dataset
from torch.nn import TransformerEncoder, TransformerEncoderLayer, LayerNorm
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaModel
import pandas as pd

class RobertaTextDataset(Dataset):
    def __init__(self, csv_file, max_length=128, device='cpu'):
        self.device = device
        self.data = pd.read_csv(csv_file)
        self.tokenizer = RobertaTokenizer.from_pretrained('/root/autodl-tmp/huggingface/roberta-base')
        self.model = RobertaModel.from_pretrained('/root/autodl-tmp/huggingface/roberta-base').to(device)

        self.max_length = max_length
        self.proj_concat = nn.Linear(3072, 1024).to(device)

        encoder_layer = TransformerEncoderLayer(d_model=1024, nhead=8).to(device)
        self.transformer = TransformerEncoder(encoder_layer, num_layers=1).to(device)
        self.norm_single = LayerNorm(1024).to(device)   # 用于每个 embedding（title、summary、synopsis）
        self.norm_concat = LayerNorm(3072).to(device)   # 用于 concat 后的多模态向量

        self.titleembedding = []
        self.summariedembedding = []
        self.synopsisembedding = []
        self.titleembeddingaftertf = []
        self.summariedembeddingaftertf = []
        self.synopsisembeddingaftertf = []
        self.multiembedding = []

        # 27类标签字段名
        self.label_columns = [
            "Drama", "Thriller", "Comedy", "Action", "Adventure", "Crime", "Romance", "Mystery", "Sci-Fi", "Fantasy",
            "Horror", "Dark Comedy", "Family", "Period Drama", "Biography", "Animation", "Romantic Comedy", "Tragedy",
            "Psychological Thriller", "Psychological Drama", "Supernatural Horror", "Slapstick", "War", "History",
            "Coming-of-Age", "Superhero", "Docudrama"
        ]

        self._preprocess_all()

    def get_model(self):
        return self.model

    def _get_embedding(self, text):
        with torch.no_grad():
            inputs = self.tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=self.max_length).to(self.device)
            outputs = self.model(**inputs)
            return outputs.last_hidden_state.mean(dim=1).squeeze(0)  # shape: (1024,)

    def _transform_and_norm(self, embedding_list):
        batch_tensor = torch.stack(embedding_list).unsqueeze(1)  # (N, 1, 1024)
        transformed = self.transformer(batch_tensor).squeeze(1)  # (N, 1024)
        return [self.norm_single(tensor) for tensor in transformed]

    def _preprocess_all(self):
        print("开始编码文本...")

        for _, row in self.data.iterrows():
            self.titleembedding.append(self._get_embedding(str(row['title'])))
            self.summariedembedding.append(self._get_embedding(str(row['summaries'])))
            self.synopsisembedding.append(self._get_embedding(str(row['synopsis'])))

        print("Transformer处理...")
        self.titleembeddingaftertf = self._transform_and_norm(self.titleembedding)
        self.summariedembeddingaftertf = self._transform_and_norm(self.summariedembedding)
        self.synopsisembeddingaftertf = self._transform_and_norm(self.synopsisembedding)

        print("生成最终多模态embedding...")
        for t, s, y in zip(self.titleembeddingaftertf, self.summariedembeddingaftertf, self.synopsisembeddingaftertf):
            concat = torch.cat([t, s, y], dim=0)  # shape: (3072,)
            reduced = self.proj_concat(concat)    # shape: (1024,)
            self.multiembedding.append(self.norm_single(reduced))  # norm over 1024
        self.labels = torch.tensor(self.data[self.label_columns].values, dtype=torch.float32).to(self.device)
        print("预处理完成.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]

        # 从缓存或重新计算得到 embedding
        title_embedding = torch.tensor(self.titleembedding[idx])           # (1024,)
        summaried_embedding = torch.tensor(self.summariedembedding[idx])   # (1024,)
        synopsis_embedding = torch.tensor(self.synopsisembedding[idx])     # (1024,)
        title_tf = torch.tensor(self.titleembeddingaftertf[idx])                          # (1024,)
        summaried_tf = torch.tensor(self.summariedembeddingaftertf[idx])                  # (1024,)
        synopsis_tf = torch.tensor(self.synopsisembeddingaftertf[idx])                    # (1024,)
        multi_embedding = torch.tensor(self.multiembedding[idx])            # (1024,)

        # 选 7 个组成最终 tensor
        emb_list = [
            title_embedding,
            summaried_embedding,
            synopsis_embedding,
            title_tf,
            summaried_tf,
            synopsis_tf,
            multi_embedding
        ]
        embeddings = torch.stack(emb_list)   # (6, 1024)

        label = torch.tensor(self.labels[idx], dtype=torch.float32)  # (27,)

        return {
            'embeddings': embeddings,
            'labels': label
        }

# 模型:
import torch
import torch.nn as nn
from transformers import RobertaModel

class RobertaMultiInputMultiLabelClassifier(nn.Module):

    def __init__(self, model , num_labels=27):
        super().__init__()
        self.roberta = model
        self.hidden_size = self.roberta.config.hidden_size
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.hidden_size * 7, num_labels)
        self.output_dim = self.roberta.config.hidden_size

    def forward(self, embeddings):
        x = embeddings.view(embeddings.size(0), -1)   # (batch, 7*1024)
        x = self.dropout(x)
        return self.classifier(x)
# 训练脚本:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import os
from torch.optim import AdamW

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from Model.RobertaMultiInputMultiLabelClassifier import RobertaMultiInputMultiLabelClassifier
from TorchDataset.RobertaMultiInputDataset import RobertaMultiInputDataset


def collate_fn(batch):
    all_inputs = []
    all_labels = []

    for item in batch:
        # 拼接6个embedding向量 → 每个样本为 (7, 1024)
        emb_list = [
            item['title_embedding'],
            item['summaried_embedding'],
            item['synopsis_embedding'],
            item['title_tf'],
            item['summaried_tf'],
            item['synopsis_tf'],
            item['multi_embedding']
        ]
        emb_tensor = torch.stack(emb_list)
        all_inputs.append(emb_tensor)

        # 使用真实的 27 维标签向量
        all_labels.append(item['label'])

    inputs_tensor = torch.stack(all_inputs)  # (batch, 6, 1024)
    labels_tensor = torch.stack(all_labels)  # (batch, 27)

    return inputs_tensor, labels_tensor


def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_inputs, batch_labels in dataloader:
            logits = model(batch_inputs)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu())
            all_labels.append(batch_labels.cpu())

    preds = torch.cat(all_preds, dim=0)
    labels = torch.cat(all_labels, dim=0)
    report = classification_report(labels, preds, zero_division=0, output_dict=True)
    acc = (preds == labels).float().mean(dim=0).mean().item()

    print(f"✅ Val Accuracy: {acc:.4f}")
    return acc

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = RobertaMultiInputDataset('/root/autodl-tmp/MovieLabeling/datasets/train_data.csv')
    val_dataset = RobertaMultiInputDataset('/root/autodl-tmp/MovieLabeling/datasets/test_data.csv')
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=8, collate_fn=collate_fn)

    pretrainmodel = train_dataset.get_model()

    model = RobertaMultiInputMultiLabelClassifier( pretrainmodel ).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    criterion = nn.BCEWithLogitsLoss()

    best_acc = 0.0
    os.makedirs("../weights", exist_ok=True)

    for epoch in range(1, 200+1):
        model.train()
        total_loss = 0

        for batch_inputs, batch_labels in train_loader:
            optimizer.zero_grad()
            logits = model(batch_inputs)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"[Epoch {epoch}] Train Loss: {avg_loss:.4f}")

        val_acc = evaluate(model, val_loader, device)
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"../weights/best_model_sr.pt")
            print(f"✅ Saved best model (epoch {epoch}, acc {val_acc:.4f})")

if __name__ == "__main__":
train()

# 测试脚本:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import os
from torch.optim import AdamW

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from Model.RobertaMultiInputMultiLabelClassifier import RobertaMultiInputMultiLabelClassifier
from TorchDataset.RobertaMultiInputDataset import RobertaMultiInputDataset


def collate_fn(batch):
    all_inputs = []
    all_labels = []

    for item in batch:
        # 拼接6个embedding向量 → 每个样本为 (7, 1024)
        emb_list = [
            item['title_embedding'],
            item['summaried_embedding'],
            item['synopsis_embedding'],
            item['title_tf'],
            item['summaried_tf'],
            item['synopsis_tf'],
            item['multi_embedding']
        ]
        emb_tensor = torch.stack(emb_list)
        all_inputs.append(emb_tensor)

        # 使用真实的 27 维标签向量
        all_labels.append(item['label'])

    inputs_tensor = torch.stack(all_inputs)  # (batch, 6, 1024)
    labels_tensor = torch.stack(all_labels)  # (batch, 27)

    return inputs_tensor, labels_tensor


def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_inputs, batch_labels in dataloader:
            logits = model(batch_inputs)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu())
            all_labels.append(batch_labels.cpu())

    preds = torch.cat(all_preds, dim=0)
    labels = torch.cat(all_labels, dim=0)
    # sklearn 自带报告
    report = classification_report(labels, preds, zero_division=0, output_dict=True)

    # 计算准确率
    acc = (preds == labels.int()).float().mean().item()

    # 🔹 调用自定义评估函数
    extra_metrics = evaluate_metrics(labels, preds)

    print(f"✅ Val Accuracy: {acc:.4f}")
    print("Extra metrics:", extra_metrics)   # 额外指标输出
    return acc

def test():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = RobertaMultiInputDataset('/root/autodl-tmp/MovieLabeling/datasets/train_data.csv')
    val_dataset = RobertaMultiInputDataset('/root/autodl-tmp/MovieLabeling/datasets/test_data.csv')
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=8, collate_fn=collate_fn)

    pretrainmodel = train_dataset.get_model()

    model = RobertaMultiInputMultiLabelClassifier( pretrainmodel ).to(device)

    val_acc = evaluate(model, val_loader, device)

if __name__ == "__main__":
    test()


# DeBERTa部分:
# 数据部分:
import torch
from torch.utils.data import Dataset
from torch.nn import TransformerEncoder, TransformerEncoderLayer, LayerNorm
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import pandas as pd

class DebertaTextDataset(Dataset):
    def __init__(self, csv_file, max_length=128, device='cpu'):
        self.device = device
        self.data = pd.read_csv(csv_file)
        self.tokenizer = AutoTokenizer.from_pretrained('/root/autodl-tmp/huggingface/microsoft/deberta-v3-base')
        self.model = AutoModel.from_pretrained('/root/autodl-tmp/huggingface/microsoft/deberta-v3-base').to(device)

        self.max_length = max_length
        self.proj_concat = nn.Linear(3072, 1024).to(device)

        encoder_layer = TransformerEncoderLayer(d_model=1024, nhead=8).to(device)
        self.transformer = TransformerEncoder(encoder_layer, num_layers=1).to(device)
        self.norm_single = LayerNorm(1024).to(device)   # 用于每个 embedding（title、summary、synopsis）
        self.norm_concat = LayerNorm(3072).to(device)   # 用于 concat 后的多模态向量

        self.titleembedding = []
        self.summariedembedding = []
        self.synopsisembedding = []
        self.titleembeddingaftertf = []
        self.summariedembeddingaftertf = []
        self.synopsisembeddingaftertf = []
        self.multiembedding = []

        # 27类标签字段名
        self.label_columns = [
            "Drama", "Thriller", "Comedy", "Action", "Adventure", "Crime", "Romance", "Mystery", "Sci-Fi", "Fantasy",
            "Horror", "Dark Comedy", "Family", "Period Drama", "Biography", "Animation", "Romantic Comedy", "Tragedy",
            "Psychological Thriller", "Psychological Drama", "Supernatural Horror", "Slapstick", "War", "History",
            "Coming-of-Age", "Superhero", "Docudrama"
        ]

        self._preprocess_all()

    def get_model(self):
        return self.model

    def _get_embedding(self, text):
        with torch.no_grad():
            inputs = self.tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=self.max_length).to(self.device)
            outputs = self.model(**inputs)
            return outputs.last_hidden_state.mean(dim=1).squeeze(0)

    def _transform_and_norm(self, embedding_list):
        batch_tensor = torch.stack(embedding_list).unsqueeze(1)
        transformed = self.transformer(batch_tensor).squeeze(1)
        return [self.norm_single(tensor) for tensor in transformed]

    def _preprocess_all(self):
        print("开始编码文本...")

        for _, row in self.data.iterrows():
            self.titleembedding.append(self._get_embedding(str(row['title'])))
            self.summariedembedding.append(self._get_embedding(str(row['summaries'])))
            self.synopsisembedding.append(self._get_embedding(str(row['synopsis'])))

        print("Transformer处理...")
        self.titleembeddingaftertf = self._transform_and_norm(self.titleembedding)
        self.summariedembeddingaftertf = self._transform_and_norm(self.summariedembedding)
        self.synopsisembeddingaftertf = self._transform_and_norm(self.synopsisembedding)

        print("生成最终多模态embedding...")
        for t, s, y in zip(self.titleembeddingaftertf, self.summariedembeddingaftertf, self.synopsisembeddingaftertf):
            concat = torch.cat([t, s, y], dim=0)
            reduced = self.proj_concat(concat)
            self.multiembedding.append(self.norm_single(reduced))  # norm over 1024
        self.labels = torch.tensor(self.data[self.label_columns].values, dtype=torch.float32).to(self.device)
        print("预处理完成.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]

        # 从缓存或重新计算得到 embedding
        title_embedding = torch.tensor(self.titleembedding[idx])
        summaried_embedding = torch.tensor(self.summariedembedding[idx])
        synopsis_embedding = torch.tensor(self.synopsisembedding[idx])
        title_tf = torch.tensor(self.titleembeddingaftertf[idx])
        summaried_tf = torch.tensor(self.summariedembeddingaftertf[idx])
        synopsis_tf = torch.tensor(self.synopsisembeddingaftertf[idx])
        multi_embedding = torch.tensor(self.multiembedding[idx])

        # 选 7 个组成最终 tensor
        emb_list = [
            title_embedding,
            summaried_embedding,
            synopsis_embedding,
            title_tf,
            summaried_tf,
            synopsis_tf,
            multi_embedding
        ]
        embeddings = torch.stack(emb_list)   # (6, 1024)

        label = torch.tensor(self.labels[idx], dtype=torch.float32)  # (27,)

        return {
            'embeddings': embeddings,
            'labels': label
        }

# 模型:
import torch
import torch.nn as nn
from transformers import AutoModel  # 使用 AutoModel 兼容性更高

class DebertaMultiInputMultiLabelClassifier(nn.Module):
    def __init__(self, model_name='microsoft/deberta-v3-base', num_labels=27):
        super().__init__()
        self.encoder = AutoModel .from_pretrained( model_name )
        self.hidden_size = self.encoder.config.d_model  # 注意不是 hidden_size，而是 d_model
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.hidden_size * 3, num_labels)
        self.output_dim = self.encoder.config.hidden_size

    def forward(self,  ):
        x = embeddings.view(embeddings.size(0), -1)   # (batch, 7*1024)
        x = self.dropout(x)
        return self.classifier(x)

# 训练脚本：
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import os
from torch.optim import AdamW

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from Model.DebertaMultiInputMultiLabelClassifier import DebertaMultiInputMultiLabelClassifier
from TorchDataset.DebertaMultiInputDataset import DebertaMultiInputDataset

def collate_fn(batch):
    all_inputs = []
    all_labels = []

    for item in batch:
        # 拼接7个embedding向量 → 每个样本为 (7, 1024)
        emb_list = [
            item['title_embedding'],
            item['summaried_embedding'],
            item['synopsis_embedding'],
            item['title_tf'],
            item['summaried_tf'],
            item['synopsis_tf'],
            item['multi_embedding']
        ]
        emb_tensor = torch.stack(emb_list)
        all_inputs.append(emb_tensor)

        # 使用真实的 27 维标签向量
        all_labels.append(item['label'])

    inputs_tensor = torch.stack(all_inputs)  # (batch, 6, 1024)
    labels_tensor = torch.stack(all_labels)  # (batch, 27)

    return inputs_tensor, labels_tensor

def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_inputs, batch_labels in dataloader:
            logits = model(batch_inputs)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu())
            all_labels.append(batch_labels.cpu())

    preds = torch.cat(all_preds, dim=0)
    labels = torch.cat(all_labels, dim=0)
    report = classification_report(labels, preds, zero_division=0, output_dict=True)
    acc = (preds == labels).float().mean(dim=0).mean().item()

    print(f"✅ Val Accuracy: {acc:.4f}")
    return acc

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = DebertaMultiInputDataset('/root/autodl-tmp/MovieLabeling/datasets/train_data.csv')
    val_dataset = DebertaMultiInputDataset('/root/autodl-tmp/MovieLabeling/datasets/test_data.csv')
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=8, collate_fn=collate_fn)

    model = DebertaMultiInputMultiLabelClassifier().to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    criterion = nn.BCEWithLogitsLoss()

    best_acc = 0.0
    os.makedirs("../weights", exist_ok=True)

    for epoch in range(1, 200+1):
        model.train()
        total_loss = 0

        for batch_inputs, batch_labels in train_loader:
            optimizer.zero_grad()
            logits = model(batch_inputs)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"[Epoch {epoch}] Train Loss: {avg_loss:.4f}")

        val_acc = evaluate(model, val_loader, device)
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"../weights/best_model_sd.pt")
            print(f"✅ Saved best model (epoch {epoch}, acc {val_acc:.4f})")

if __name__ == "__main__":
    train()

# 测试脚本：
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import os
from torch.optim import AdamW

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from Model.DebertaMultiInputMultiLabelClassifier import DebertaMultiInputMultiLabelClassifier
from TorchDataset.DebertaMultiInputDataset import DebertaMultiInputDataset
from metrixCaler import evaluate_metrics

def collate_fn(batch):
    all_inputs = []
    all_labels = []

    for item in batch:
        # 拼接7个embedding向量 → 每个样本为 (7, 1024)
        emb_list = [
            item['title_embedding'],
            item['summaried_embedding'],
            item['synopsis_embedding'],
            item['title_tf'],
            item['summaried_tf'],
            item['synopsis_tf'],
            item['multi_embedding']
        ]
        emb_tensor = torch.stack(emb_list)
        all_inputs.append(emb_tensor)

        # 使用真实的 27 维标签向量
        all_labels.append(item['label'])

    inputs_tensor = torch.stack(all_inputs)  # (batch, 6, 1024)
    labels_tensor = torch.stack(all_labels)  # (batch, 27)

    return inputs_tensor, labels_tensor

def load_model(model_path, device='cuda'):
    # 初始化模型结构（必须与保存时一致）
    model = DebertaMultiInputMultiLabelClassifier()
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"✅ 模型已加载: {model_path}")
    return model

def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_inputs, batch_labels in dataloader:
            logits = model(batch_inputs)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu())
            all_labels.append(batch_labels.cpu())

    preds = torch.cat(all_preds, dim=0)
    labels = torch.cat(all_labels, dim=0)
     # sklearn 自带报告
    report = classification_report(labels, preds, zero_division=0, output_dict=True)

    # 计算准确率
    acc = (preds == labels.int()).float().mean().item()

    # 🔹 调用自定义评估函数
    extra_metrics = evaluate_metrics(labels, preds)

    print(f"✅ Val Accuracy: {acc:.4f}")
    print("Extra metrics:", extra_metrics)   # 额外指标输出
    return acc

def test():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = DebertaMultiInputDataset('/root/autodl-tmp/MovieLabeling/datasets/train_data.csv')
    val_dataset = DebertaMultiInputDataset('/root/autodl-tmp/MovieLabeling/datasets/test_data.csv')
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=8, collate_fn=collate_fn)

    model = DebertaMultiInputMultiLabelClassifier().to(device)
    model = load_model("../weights/best_model_d.pt", device)
    val_acc = evaluate(model, val_loader, device)

if __name__ == "__main__":
    test()


# 综合模块
# 数据部分：
import torch
from torch.utils.data import Dataset
from TorchDataset.RobertaTextDataset import RobertaTextDataset
from TorchDataset.RobertaMultiInputDataset import RobertaMultiInputDataset
from TorchDataset.DebertaMultiInputDataset import DebertaMultiInputDataset


class EnsembleDataset(Dataset):
    def __init__(self, csv_file, device='cuda'):
        self.lstmdataset = RobertaTextDataset(csv_file=csv_file)
        self.robertadataset = RobertaMultiInputDataset(csv_file=csv_file)
        self.debertadataset = DebertaMultiInputDataset(csv_file=csv_file)

        # 统一长度
        assert len(self.lstmdataset) == len(self.robertadataset) == len(self.debertadataset), "Dataset size mismatch"

    def __len__(self):
        return len(self.lstmdataset)

    def __getitem__(self, idx):
        # 获取三路输入
        lstm_item = self.lstmdataset[idx]
        #lstm_input = lstm_item['embeddings']   # (6, 1024)
        roberta_item = self.robertadataset[idx]   # 包含: input_ids, attention_mask, labels
        deberta_item = self.debertadataset[idx]   # 同上

        # 假设三个 Dataset 返回的标签是一致的，只取一个
        return {
            'lstm_input': lstm_item['embeddings'],     # shape (7, 1024)
            'roberta_input': {
                'roberta_input': roberta_item['embeddings'],
            },
            'deberta_input': {
                'deberta_input': deberta_item['embeddings'],
            },
            'labels': lstm_item['labels']
        }

# 模型部分:
import torch
import torch.nn as nn

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))


from BILSTMModel import BiLSTMClassifier
from RobertaMultiInputMultiLabelClassifier import RobertaMultiInputMultiLabelClassifier
from DebertaMultiInputMultiLabelClassifier import DebertaMultiInputMultiLabelClassifier

class EnsembleInferenceModel(nn.Module):
    '''
        :param bilstm_path : bilstm model weights
        :param roberta_path : roberta model weights
        :param deberta_path : deberta model weights
        :param model_type :
            0 : run with bilstm model \ roberta model \ deberta model
            1 : run with roberta model \ deberta model
            2 : run with roberta model \ bilstm model
            3 : run with deberta model \ bilstm model
        ....

    '''
    def __init__(self, bilstm_path, roberta_path, deberta_path, model_type=0, hidden_size=512, num_labels=27, device='cuda'):
        super().__init__()
        self.device = device

        # 加载三个子模型
        self.bilstm = BiLSTMClassifier  # 确保结构一致
        self.bilstm.load_state_dict(torch.load(bilstm_path, map_location=device))
        self.bilstm.to(device)
        self.bilstm.eval()

        self.roberta = RobertaMultiInputMultiLabelClassifier
        self.roberta.load_state_dict(torch.load(roberta_path, map_location=device))
        self.roberta.to(device)
        self.roberta.eval()

        self.deberta = DebertaMultiInputMultiLabelClassifier
        self.deberta.load_state_dict(torch.load(deberta_path, map_location=device))
        self.deberta.to(device)
        self.deberta.eval()

        self.model_type = model_type

        total_hidden = self.bilstm.output_dim + self.roberta.output_dim + self.deberta.output_dim
        if self.model_type == 1:
            # 获取子模型输出维度
            total_hidden = self.roberta.output_dim + self.deberta.output_dim
        if self.model_type == 2:
            # 获取子模型输出维度
            total_hidden = self.bilstm.output_dim + self.roberta.output_dim
        if self.model_type == 42:
            # 获取子模型输出维度
            total_hidden =self.deberta.output_dim + self.bilstm.output_dim


        # MLP 分类器
        self.classifier = nn.Sequential(
            nn.Linear(total_hidden, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, num_labels)
        )

    def forward(self, inputs_bilstm, inputs_roberta, inputs_deberta):
        with torch.no_grad():
            vec1 = self.bilstm(inputs_bilstm)
            vec2 = self.roberta(inputs_roberta)
            vec3 = self.deberta(inputs_deberta)

        fused = torch.cat([vec1, vec2, vec3], dim=1)
        if self.model_type == 1:
            # 获取子模型输出维度
            fused = torch.cat([vec2, vec3], dim=1)
        if self.model_type == 2:
            # 获取子模型输出维度
            fused = torch.cat([vec2, vec1], dim=1)
        if self.model_type == 42:
            # 获取子模型输出维度
            fused = torch.cat([vec3, vec1], dim=1)
        logits = self.classifier(fused)
        print('logits')
        print(logits)
        return logits

# 训练脚本:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import logging

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from Model.BILSTMModel import BiLSTMClassifier
from Model.RobertaMultiInputMultiLabelClassifier import RobertaMultiInputMultiLabelClassifier
from Model.DebertaMultiInputMultiLabelClassifier import DebertaMultiInputMultiLabelClassifier

from TorchDataset.EnsembleDataset import EnsembleDataset  # 你需实现此类
from sklearn.metrics import classification_report

logging.set_verbosity_error()

def load_model(path='bilstm_model.pt', device='cpu'):
    checkpoint = torch.load(path, map_location=device)

    if checkpoint['model_class'] == 'BiLSTMClassifier':
        model = BiLSTMClassifier(num_classes=27).to(device)
    else:
        raise ValueError(f"未知模型类型: {checkpoint['model_class']}")

    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"模型已从 {path} 加载")
    return model

# === 集成模型 === #
class EnsembleClassifier(nn.Module):
    def __init__(self, bilstm, roberta, deberta, hidden_dim=512, num_labels=27):
        super().__init__()
        self.bilstm = bilstm.eval()
        self.roberta = roberta.eval()
        self.deberta = deberta.eval()

        self.bilstm_output_dim = 27# bilstm.output_dim
        self.roberta_output_dim = 27# roberta.output_dim
        self.deberta_output_dim = 27# deberta.output_dim

        total_dim = self.bilstm_output_dim + self.roberta_output_dim + self.deberta_output_dim
        self.classifier = nn.Sequential(
            nn.Linear(total_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_labels)
        )

        # 冻结三个模型
        for p in self.bilstm.parameters(): p.requires_grad = False
        for p in self.roberta.parameters(): p.requires_grad = False
        for p in self.deberta.parameters(): p.requires_grad = False

    def forward(self, bilstm_input, roberta_input, deberta_input):
        with torch.no_grad():
            v1 = self.bilstm(bilstm_input)  # (B, D1)
            v2 = self.roberta(roberta_input['roberta_input'])  # (B, D2)
            v3 = self.deberta(deberta_input['deberta_input'])  # (B, D3
        fused = torch.cat([v1, v2, v3], dim=1)
        logits = self.classifier(fused)
        return logits


# === 训练函数 === #
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in dataloader:
        bilstm_input = batch['lstm_input'].to(device)
        roberta_input = {
        'roberta_input': {k: v.to(device) for k, v in batch['roberta_input'].items()},
        }
        deberta_input = {
            'deberta_input': {k: v.to(device) for k, v in batch['deberta_input'].items()},
        }
        labels = batch['labels'].float().to(device)

        logits = model(bilstm_input, roberta_input, deberta_input)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


# === 验证函数 === #
def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            bilstm_input = batch['lstm_input'].to(device)
            roberta_input = {
            'roberta_input': {k: v.to(device) for k, v in batch['roberta_input'].items()},
            }
            deberta_input = {
                'deberta_input': {k: v.to(device) for k, v in batch['deberta_input'].items()},
            }
            labels = batch['labels'].float().to(device)

            logits = model(bilstm_input, roberta_input, deberta_input)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)

    report = classification_report(labels, preds, zero_division=0, output_dict=True)
    print(classification_report(labels, preds, zero_division=0))
    return report["macro avg"]["f1-score"]


# === 主函数 === #
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # === 加载子模型 === #
    bilstm = load_model('../weights/bilstm_model_0.9.pt')#BiLSTMClassifier(num_classes=27)
    #bilstm.load_state_dict(torch.load('../weights/bilstm_model_0.9.pt'))
    #bilstm.output_dim = 128  # 你必须设置此属性

    roberta = RobertaMultiInputMultiLabelClassifier()
    roberta.load_state_dict(torch.load('../weights/best_model_sr.pt'))
    #roberta.output_dim = 1024

    deberta = DebertaMultiInputMultiLabelClassifier()
    deberta.load_state_dict(torch.load('../weights/best_model_sd.pt'))
    #deberta.output_dim = 1024

    # === 构建集成模型 === #
    model = EnsembleClassifier(bilstm, roberta, deberta).to(device)

    # === 加载数据 === #
    train_dataset = EnsembleDataset(csv_file='/root/autodl-tmp/MovieLabeling/datasets/train_data.csv')
    val_dataset = EnsembleDataset(csv_file='/root/autodl-tmp/MovieLabeling/datasets/test_data.csv')
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16)

    # === 优化器和损失 === #
    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=2e-5)
    criterion = nn.BCEWithLogitsLoss()

    # === 训练过程 === #
    best_f1 = 0
    for epoch in range(20):
        train_loss = train(model, train_loader, optimizer, criterion, device)
        val_f1 = evaluate(model, val_loader, device)
        print(f"[Epoch {epoch+1}] Train Loss: {train_loss:.4f}, Val Macro F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), '../weights/best_ensemble.pt')
            print("✅ Saved new best model!")

if __name__ == '__main__':
    main()

# 测试脚本:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import logging
from sklearn.metrics import classification_report

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from Model.BILSTMModel import BiLSTMClassifier
from Model.RobertaMultiInputMultiLabelClassifier import RobertaMultiInputMultiLabelClassifier
from Model.DebertaMultiInputMultiLabelClassifier import DebertaMultiInputMultiLabelClassifier
from TorchDataset.EnsembleDataset import EnsembleDataset
from metrixCaler import evaluate_metrics


# ==== 屏蔽transformers的warning ====
logging.set_verbosity_error()


import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def plot_multiclass_roc(y_true, y_score, num_classes, save_path=None):
    """
    绘制多分类 ROC 曲线，并可保存图片。

    参数:
    y_true: 长度为N的真实标签（整数类别）
    y_score: N×C 的概率数组，C为类别数
    num_classes: 类别数量
    save_path: 如果不为 None，则保存图像到该路径（如 'roc_curve.png'）
    """
    y_test = label_binarize(y_true, classes=range(num_classes))
    y_test = np.array(y_test)
    y_score = np.array(y_score)

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    plt.figure(figsize=(10, 8))

    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], lw=2, label=f'Class {i} (AUC = {roc_auc[i]:.2f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Multiclass ROC Curve')
    plt.legend(loc='lower right', fontsize='small')
    plt.grid(True)

    # 保存图像
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✅ ROC 图已保存至: {save_path}")

    #plt.show()

# ==== 加载子模型 ====
def load_model(path='bilstm_model.pt', device='cpu'):
    checkpoint = torch.load(path, map_location=device)
    model = BiLSTMClassifier(num_classes=27).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model

def strict_accuracy(preds: torch.Tensor, labels: torch.Tensor) -> float:
    """
    计算严格准确率：所有标签都完全匹配才算正确。

    参数:
        preds (torch.Tensor): 预测结果，形状为 (batch_size, num_labels)，值为 0 或 1。
        labels (torch.Tensor): 真实标签，形状相同。

    返回:
        float: 严格准确率。
    """
    correct = (preds == labels).all(dim=1).float()  # 每一行是否全部匹配
    acc = correct.mean().item()
    return acc

# ==== 集成模型结构 ====
class EnsembleClassifier(nn.Module):
    def __init__(self, bilstm, roberta, deberta, hidden_dim=512, num_labels=27):
        super().__init__()
        self.bilstm = bilstm.eval()
        self.roberta = roberta.eval()
        self.deberta = deberta.eval()

        self.bilstm_output_dim = 27
        self.roberta_output_dim = 27
        self.deberta_output_dim = 27

        total_dim = self.bilstm_output_dim + self.roberta_output_dim + self.deberta_output_dim
        self.classifier = nn.Sequential(
            nn.Linear(total_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self, bilstm_input, roberta_input, deberta_input):
        with torch.no_grad():
            v1 = self.bilstm(bilstm_input)
            v2 = self.roberta(
                roberta_input['roberta_input']
            )
            v3 = self.deberta(
                deberta_input['deberta_input']
            )

        fused = torch.cat([v1, v2, v3], dim=1)
        logits = self.classifier(fused)

        return logits


# ==== 测试函数 ====
def test(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            bilstm_input = batch['lstm_input'].to(device)
            roberta_input = {
                'roberta_input': {k: v.to(device) for k, v in batch['roberta_input'].items()},
            }
            deberta_input = {
                'deberta_input': {k: v.to(device) for k, v in batch['deberta_input'].items()},
            }
            labels = batch['labels'].to(device)

            logits = model(bilstm_input, roberta_input, deberta_input)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    metrix = evaluate_metrics(preds,labels)
    print(metrix)

    acc = strict_accuracy(preds,labels)
    print(f'acc = {acc}')
    print("\n📊 Classification Report:")
    print(classification_report(labels, preds, zero_division=0))
    # 概率分布
    probs = torch.softmax(preds.float(), dim=1)

    # 🎯 绘制 ROC 曲线
    plot_multiclass_roc(
        y_true=labels.cpu().numpy(),
        y_score=probs.cpu().numpy(),
        num_classes=probs.size(1),
        save_path='roc_curve.png'
    )

    return preds


# ==== 主入口 ====
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 加载子模型
    bilstm = load_model('../weights/bilstm_model_0.9.pt', device=device)
    roberta = RobertaMultiInputMultiLabelClassifier()
    roberta.load_state_dict(torch.load('../weights/best_model_sr.pt'))
    deberta = DebertaMultiInputMultiLabelClassifier()
    deberta.load_state_dict(torch.load('../weights/best_model_sd.pt'))

    # 构建集成模型并加载权重
    model = EnsembleClassifier(bilstm, roberta, deberta).to(device)
    model.load_state_dict(torch.load('../weights/best_ensemble.pt', map_location=device))
    model.eval()

    # 加载测试数据
    test_dataset = EnsembleDataset(csv_file='/root/autodl-tmp/MovieLabeling/datasets/test_data.csv')
    test_loader = DataLoader(test_dataset, batch_size=16)

    # 开始测试
    test(model, test_loader, device)


if __name__ == '__main__':
    main()

